# LeetCode 480 — Sliding Window Median bằng Python

- Link: https://leetcode.com/problems/sliding-window-median/
- Độ khó: **Hard**
- Chủ đề: Sliding Window (cửa sổ trượt), Heap (hàng đợi ưu tiên), Sorted List, Binary Search (`bisect`)

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def medianSlidingWindow(self, nums: List[int], k: int) -> List[float]:

```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu **trung vị (median)** là gì, cách tính với số phần tử lẻ/chẵn, và vì sao trung vị "khó" hơn trung bình khi cửa sổ trượt.
2. Biết dùng `sorted`, `bisect.insort`, `bisect.bisect_left` để giữ một **danh sách luôn được sắp xếp**.
3. Nắm kỹ thuật **hai heap** (max-heap nửa dưới + min-heap nửa trên) và mẹo **xóa trễ (lazy deletion)** khi heap không hỗ trợ xóa phần tử bất kỳ.
4. Tự cài 4 lời giải từ chậm đến nhanh: sắp xếp lại mỗi cửa sổ → danh sách sắp xếp + `bisect` → hai heap + xóa trễ → `SortedList` (nếu có).
5. Biết cách **trace**, kiểm thử ngẫu nhiên có seed và đối chiếu với brute force.

## 2. Đề bài & ví dụ

**Phát biểu lại bằng tiếng Việt.**

Cho mảng số nguyên `nums` và số nguyên `k`. Có một **cửa sổ** độ dài `k` bắt đầu ở đầu mảng và mỗi lần trượt sang phải **một** vị trí. Với mỗi vị trí cửa sổ, hãy tính **trung vị** của `k` số trong cửa sổ. Trả về danh sách các trung vị (theo thứ tự cửa sổ từ trái sang phải). Đáp án sai số trong khoảng `10^-5` được chấp nhận.

**Trung vị** của một dãy:
- Sắp xếp dãy tăng dần.
- Nếu số phần tử **lẻ**: trung vị là phần tử ở giữa. `[2,3,4]` → `3`.
- Nếu số phần tử **chẵn**: trung vị là trung bình của hai phần tử giữa. `[2,3]` → `(2+3)/2 = 2.5`.

**Ví dụ 1**

```text
Input:  nums = [1,3,-1,-3,5,3,6,7], k = 3
Output: [1.0,-1.0,-1.0,3.0,5.0,6.0]

Cửa sổ               Trung vị
[1  3  -1] -3  5  3  6  7       1
 1 [3  -1  -3] 5  3  6  7      -1
 1  3 [-1  -3  5] 3  6  7      -1
 1  3  -1 [-3  5  3] 6  7       3
 1  3  -1  -3 [5  3  6] 7       5
 1  3  -1  -3  5 [3  6  7]      6
```

**Ví dụ 2**

```text
Input:  nums = [1,2,3,4,2,3,1,4,2], k = 3
Output: [2.0,3.0,3.0,3.0,2.0,3.0,2.0]
```

**Ràng buộc (constraints)**

- `1 <= k <= nums.length <= 10^5`
- `-2^31 <= nums[i] <= 2^31 - 1`

Nhận xét nhanh: có `n - k + 1` cửa sổ. Với `n = 10^5`, `k = 5·10^4`, nếu mỗi cửa sổ sắp xếp lại thì tốn khoảng `5·10^4 × 5·10^4 × log` phép — quá chậm. Ta cần cập nhật **từng bước nhỏ** khi cửa sổ trượt.

In [1]:
# Chạy thử: liệt kê từng cửa sổ của ví dụ 1 và trung vị (tính "bằng mắt" với sorted)
nums = [1, 3, -1, -3, 5, 3, 6, 7]
k = 3
for i in range(len(nums) - k + 1):
    window = nums[i:i + k]
    s = sorted(window)
    print(f"i={i}  window={window}  sorted={s}  median={s[k // 2]}")

i=0  window=[1, 3, -1]  sorted=[-1, 1, 3]  median=1
i=1  window=[3, -1, -3]  sorted=[-3, -1, 3]  median=-1
i=2  window=[-1, -3, 5]  sorted=[-3, -1, 5]  median=-1
i=3  window=[-3, 5, 3]  sorted=[-3, 3, 5]  median=3
i=4  window=[5, 3, 6]  sorted=[3, 5, 6]  median=5
i=5  window=[3, 6, 7]  sorted=[3, 6, 7]  median=6


## 3. Kiến thức nền

Trước khi giải, ta ôn từng công cụ Python sẽ dùng. Mỗi công cụ có một ví dụ nhỏ chạy được.

### 3.1. Slicing `nums[i:i+k]` — lấy cửa sổ

`nums[a:b]` lấy các phần tử có chỉ số `a, a+1, ..., b-1` (không lấy `b`). Vậy `nums[i:i+k]` có đúng `k` phần tử bắt đầu từ `i`.

Số cửa sổ: chỉ số bắt đầu `i` chạy từ `0` đến `n-k` → có `n-k+1` cửa sổ.

In [2]:
nums = [10, 20, 30, 40, 50]
k = 2
print("Số cửa sổ:", len(nums) - k + 1)
for i in range(len(nums) - k + 1):
    print(i, nums[i:i + k])

Số cửa sổ: 4
0 [10, 20]
1 [20, 30]
2 [30, 40]
3 [40, 50]


### 3.2. Tính trung vị từ một danh sách đã sắp xếp

Nếu `s` đã sắp xếp và có `k` phần tử:
- `k` lẻ → `s[k // 2]`.
- `k` chẵn → `(s[k // 2 - 1] + s[k // 2]) / 2`.

Một công thức gộp hay dùng: `(s[k // 2] + s[(k - 1) // 2]) / 2`. Với `k` lẻ, `k//2 == (k-1)//2` nên ra đúng phần tử giữa; với `k` chẵn ra trung bình hai phần tử giữa.

Lưu ý: phép chia `/` trong Python 3 **luôn** trả về `float`, còn `//` là chia lấy phần nguyên (làm tròn xuống).

In [3]:
def median_of_sorted(s):
    k = len(s)
    return (s[k // 2] + s[(k - 1) // 2]) / 2

print(median_of_sorted([2, 3, 4]))      # 3.0
print(median_of_sorted([2, 3]))         # 2.5
print(median_of_sorted([-5]))           # -5.0
print(7 // 2, (7 - 1) // 2, 8 // 2, (8 - 1) // 2)   # 3 3 4 3

3.0
2.5
-5.0
3 3 4 3


### 3.3. Module `bisect` — tìm vị trí và chèn vào danh sách đã sắp xếp

- `bisect.bisect_left(s, x)`: trả về chỉ số **đầu tiên** mà `s[idx] >= x`. Nếu `x` có trong `s`, đó chính là vị trí của bản sao đầu tiên của `x`. Tốn `O(log k)` (tìm kiếm nhị phân).
- `bisect.insort(s, x)`: chèn `x` vào đúng chỗ để `s` vẫn sắp xếp. Tìm chỗ mất `O(log k)` nhưng **dời phần tử** để chèn mất `O(k)`.
- Xóa một phần tử biết vị trí: `s.pop(idx)` mất `O(k)` (phải dời các phần tử phía sau).

In [4]:
import bisect
s = [1, 3, 3, 7]
print("bisect_left(s, 3) =", bisect.bisect_left(s, 3))   # 1
print("bisect_left(s, 5) =", bisect.bisect_left(s, 5))   # 3
bisect.insort(s, 5)
print("sau insort 5:", s)
idx = bisect.bisect_left(s, 3)
s.pop(idx)
print("sau khi xóa một số 3:", s)

bisect_left(s, 3) = 1
bisect_left(s, 5) = 3
sau insort 5: [1, 3, 3, 5, 7]
sau khi xóa một số 3: [1, 3, 5, 7]


### 3.4. Module `heapq` — min-heap, và mẹo đổi dấu để có max-heap

**Heap** (đống) là cấu trúc cho phép:
- Lấy phần tử **nhỏ nhất** ở đỉnh: `h[0]` — `O(1)`.
- Thêm phần tử: `heapq.heappush(h, x)` — `O(log n)`.
- Lấy ra phần tử nhỏ nhất: `heapq.heappop(h)` — `O(log n)`.

Python chỉ có **min-heap**. Muốn **max-heap** (đỉnh là số lớn nhất), ta lưu **số đối** `-x`: số lớn nhất trở thành số âm nhỏ nhất.

Điểm yếu quan trọng: heap **không** hỗ trợ xóa một phần tử bất kỳ ở giữa một cách nhanh. Ta sẽ xử lý bằng **xóa trễ** ở mục 6.

In [5]:
import heapq
h = []
for x in [5, 1, 8, 3]:
    heapq.heappush(h, x)
print("min-heap đỉnh:", h[0])          # 1

mx = []
for x in [5, 1, 8, 3]:
    heapq.heappush(mx, -x)
print("max-heap đỉnh:", -mx[0])        # 8
print("pop max:", -heapq.heappop(mx), "đỉnh mới:", -mx[0])

min-heap đỉnh: 1
max-heap đỉnh: 8
pop max: 8 đỉnh mới: 5


### 3.5. `collections.defaultdict(int)` — bộ đếm "chờ xóa"

`defaultdict(int)` là dict mà khi truy cập khóa chưa có thì tự tạo giá trị `0`. Rất tiện để đếm "số `x` đang chờ bị xóa": `delayed[x] += 1`.

In [6]:
from collections import defaultdict
delayed = defaultdict(int)
delayed[4] += 1
delayed[4] += 1
delayed[-2] += 1
print(dict(delayed), "delayed[100] =", delayed[100])

{4: 2, -2: 1} delayed[100] = 0


## 4. Ý tưởng nền tảng / trực giác

Hãy tính tay ví dụ nhỏ `nums = [5, 2, 8, 1, 9]`, `k = 3`.

| Cửa sổ | Sắp xếp | Trung vị |
|---|---|---|
| `[5,2,8]` | `[2,5,8]` | 5 |
| `[2,8,1]` | `[1,2,8]` | 2 |
| `[8,1,9]` | `[1,8,9]` | 8 |

Quan sát: từ cửa sổ 1 sang cửa sổ 2, chỉ có **một số rời đi** (`5`) và **một số đi vào** (`1`). Hai số còn lại (`2, 8`) giữ nguyên. Vậy thay vì sắp xếp lại từ đầu, ta chỉ cần:

1. **Xóa** số rời đi khỏi cấu trúc dữ liệu.
2. **Thêm** số đi vào.
3. **Đọc** trung vị.

Câu hỏi then chốt: *Cấu trúc dữ liệu nào cho phép thêm, xóa, đọc phần tử giữa đều nhanh?* Đó là chủ đề của cả notebook.

Một trực giác nữa: trung vị chia dãy thành **nửa nhỏ** và **nửa lớn**. Nếu ta giữ được "số lớn nhất của nửa nhỏ" và "số nhỏ nhất của nửa lớn" thì trung vị đọc ra ngay. Đây là gốc rễ của lời giải hai heap.

In [7]:
nums = [5, 2, 8, 1, 9]
k = 3
for i in range(len(nums) - k + 1):
    out_num = nums[i - 1] if i > 0 else None
    in_num = nums[i + k - 1]
    w = sorted(nums[i:i + k])
    print(f"cửa sổ {i}: rời đi={out_num}, đi vào={in_num}, sorted={w}, median={w[k//2]}")

cửa sổ 0: rời đi=None, đi vào=8, sorted=[2, 5, 8], median=5
cửa sổ 1: rời đi=5, đi vào=1, sorted=[1, 2, 8], median=2
cửa sổ 2: rời đi=2, đi vào=9, sorted=[1, 8, 9], median=8


## 5. Cách 1 — Brute Force: sắp xếp lại từng cửa sổ

**Ý tưởng.** Với mỗi `i` từ `0` đến `n-k`, cắt `nums[i:i+k]`, sắp xếp, lấy trung vị. Đúng theo định nghĩa nên rất dễ tin.

**Các bước:**
1. Tạo danh sách kết quả rỗng.
2. Với mỗi cửa sổ: `s = sorted(nums[i:i+k])`.
3. Thêm `median_of_sorted(s)` vào kết quả.

In [8]:
def median_brute(nums, k):
    res = []
    for i in range(len(nums) - k + 1):
        s = sorted(nums[i:i + k])
        res.append((s[k // 2] + s[(k - 1) // 2]) / 2)
    return res

print(median_brute([1, 3, -1, -3, 5, 3, 6, 7], 3))
print(median_brute([1, 2, 3, 4, 2, 3, 1, 4, 2], 3))
assert median_brute([1, 3, -1, -3, 5, 3, 6, 7], 3) == [1.0, -1.0, -1.0, 3.0, 5.0, 6.0]
assert median_brute([1, 2, 3, 4, 2, 3, 1, 4, 2], 3) == [2.0, 3.0, 3.0, 3.0, 2.0, 3.0, 2.0]
assert median_brute([1, 4, 2, 3], 4) == [2.5]

[1.0, -1.0, -1.0, 3.0, 5.0, 6.0]
[2.0, 3.0, 3.0, 3.0, 2.0, 3.0, 2.0]


**Độ phức tạp.**
- Time: `O((n-k+1) · k log k)`. Với `k ≈ n/2 = 5·10^4` thì khoảng `5·10^4 · 5·10^4 · 16 ≈ 4·10^10` — không thể chạy kịp.
- Space: `O(k)` cho bản sao cửa sổ.

**Nút thắt.** Mỗi lần trượt ta vứt bỏ toàn bộ thứ tự đã biết rồi sắp xếp lại. Trong khi đó chỉ có **2 phần tử thay đổi**. Ta lãng phí thông tin.

In [9]:
import time, random
random.seed(1)
big = [random.randint(-10**9, 10**9) for _ in range(4000)]
t = time.perf_counter()
median_brute(big, 2000)
print(f"Brute force n=4000, k=2000: {time.perf_counter() - t:.3f} giây (n=10^5 sẽ lâu hơn hàng nghìn lần)")

Brute force n=4000, k=2000: 0.322 giây (n=10^5 sẽ lâu hơn hàng nghìn lần)


## 6. Bước đột phá

**Insight 1 — Giữ cửa sổ luôn ở dạng đã sắp xếp.** Nếu `window` đã sắp xếp, khi trượt:
- Xóa `out = nums[i-k]`: tìm vị trí bằng `bisect_left` rồi `pop`.
- Thêm `inn = nums[i]`: `insort`.
- Trung vị đọc trực tiếp bằng chỉ số `k//2`.

Bất biến (invariant): *sau mỗi bước, `window` là đúng `k` số của cửa sổ hiện tại, sắp xếp tăng dần.*

**Insight 2 — Không cần thứ tự toàn bộ, chỉ cần "ranh giới giữa".** Chia cửa sổ thành:
- `lo` (nửa nhỏ) — **max-heap**, đỉnh là số lớn nhất của nửa nhỏ.
- `hi` (nửa lớn) — **min-heap**, đỉnh là số nhỏ nhất của nửa lớn.

Bất biến:
1. Mọi số trong `lo` `<=` mọi số trong `hi`.
2. `size(lo) == size(hi)` hoặc `size(lo) == size(hi) + 1` (nửa nhỏ được hơn 1 khi `k` lẻ).

Khi đó: **nếu** `k` lẻ **thì** trung vị = đỉnh `lo`; **nếu** `k` chẵn **thì** trung vị = (đỉnh `lo` + đỉnh `hi`) / 2.

**Insight 3 — Xóa trễ (lazy deletion).** Heap không xóa được phần tử ở giữa. Mẹo: khi số `out` rời cửa sổ, ta **chỉ ghi chú** `delayed[out] += 1` và **cập nhật kích thước logic** của heap chứa nó. Phần tử thật sự chỉ bị loại khi nó **trồi lên đỉnh** (lúc đó ta pop nó và giảm `delayed`). Quy tắc: **nếu** đỉnh heap là số đang chờ xóa **thì** pop ngay, lặp lại tới khi đỉnh "sạch".

Vì sao xóa trễ an toàn? Vì ta chỉ bao giờ **đọc đỉnh**. Rác nằm sâu bên trong không ảnh hưởng gì, miễn là trước khi đọc đỉnh ta dọn rác ở đỉnh.

Làm sao biết `out` nằm trong `lo` hay `hi`? So với đỉnh `lo`: **nếu** `out <= -lo[0]` **thì** nó thuộc `lo` (về mặt logic), ngược lại thuộc `hi`. Khi có số trùng nhau, quy tắc này vẫn đúng vì ta chỉ cần trừ đúng *kích thước* của nửa chứa "một bản" của giá trị đó — các bản giống nhau có thể đổi chỗ cho nhau mà không thay đổi trung vị.

In [10]:
# Minh họa Insight 2: chia [2,5,8,1] thành hai nửa
import heapq
vals = sorted([2, 5, 8, 1])
lo = [-x for x in vals[:2]]; heapq.heapify(lo)   # nửa nhỏ: {1,2} -> max-heap
hi = vals[2:]; heapq.heapify(hi)                  # nửa lớn: {5,8} -> min-heap
print("đỉnh lo =", -lo[0], " đỉnh hi =", hi[0], " median =", (-lo[0] + hi[0]) / 2)

đỉnh lo = 2  đỉnh hi = 5  median = 3.5


## 7. Các cách tiếp theo

### 7.1. Cách 2 — Danh sách sắp xếp + `bisect`

**Giải thích.** Khởi tạo `window = sorted(nums[:k])`. Với mỗi `i` từ `k` đến `n-1`: ghi trung vị hiện tại, xóa `nums[i-k]`, chèn `nums[i]`. Cuối cùng ghi trung vị của cửa sổ cuối.

Chi phí mỗi bước: tìm vị trí `O(log k)`, nhưng `pop`/`insort` phải dời phần tử `O(k)`. Tổng `O(n·k)` về lý thuyết; trong thực tế việc dời mảng trong C rất nhanh (dùng `memmove`) nên chạy được trên LeetCode.

In [11]:
import bisect

def median_bisect(nums, k):
    window = sorted(nums[:k])
    def med():
        return (window[k // 2] + window[(k - 1) // 2]) / 2
    res = [med()]
    for i in range(k, len(nums)):
        out = nums[i - k]
        window.pop(bisect.bisect_left(window, out))   # xóa số rời đi
        bisect.insort(window, nums[i])                # thêm số đi vào
        res.append(med())
    return res

print(median_bisect([1, 3, -1, -3, 5, 3, 6, 7], 3))
assert median_bisect([1, 3, -1, -3, 5, 3, 6, 7], 3) == [1.0, -1.0, -1.0, 3.0, 5.0, 6.0]
assert median_bisect([1, 2, 3, 4, 2, 3, 1, 4, 2], 3) == [2.0, 3.0, 3.0, 3.0, 2.0, 3.0, 2.0]

[1.0, -1.0, -1.0, 3.0, 5.0, 6.0]


**Code nộp LeetCode (Cách 2):**

In [12]:
from typing import List
import bisect

class SolutionBisect:
    def medianSlidingWindow(self, nums: List[int], k: int) -> List[float]:
        window = sorted(nums[:k])
        res = [(window[k // 2] + window[(k - 1) // 2]) / 2]
        for i in range(k, len(nums)):
            window.pop(bisect.bisect_left(window, nums[i - k]))
            bisect.insort(window, nums[i])
            res.append((window[k // 2] + window[(k - 1) // 2]) / 2)
        return res

print(SolutionBisect().medianSlidingWindow([1, 3, -1, -3, 5, 3, 6, 7], 3))

[1.0, -1.0, -1.0, 3.0, 5.0, 6.0]


**Độ phức tạp Cách 2:** Time `O(k log k + (n-k)·k)` (phần `k` là dời mảng, hằng số rất nhỏ); Space `O(k)`.

### 7.2. Cách 3 — Hai heap + xóa trễ (lời giải chuẩn `O(n log k)`)

**Giải thích từng thành phần.**

- `lo`: max-heap (lưu số đối), `hi`: min-heap.
- `lo_size`, `hi_size`: kích thước **logic** (không tính rác chờ xóa).
- `delayed`: đếm số lần mỗi giá trị đang chờ xóa.
- `prune(heap)`: dọn rác trên đỉnh của một heap.
- `balance()`: nếu `lo_size > hi_size + 1` thì chuyển đỉnh `lo` sang `hi`; nếu `lo_size < hi_size` thì chuyển đỉnh `hi` sang `lo`. Sau khi chuyển, dọn đỉnh heap vừa bị lấy.
- `add(x)`: nếu `lo` rỗng hoặc `x <= đỉnh lo` thì vào `lo`, ngược lại vào `hi`; rồi `balance()`.
- `remove(x)`: `delayed[x] += 1`; xác định `x` thuộc nửa nào bằng cách so với đỉnh `lo`; giảm size tương ứng; nếu `x` đúng là đỉnh thì dọn ngay; rồi `balance()`.

Ta viết thành một lớp `DualHeap` cho dễ đọc.

In [13]:
import heapq
from collections import defaultdict

class DualHeap:
    def __init__(self, k):
        self.k = k
        self.lo = []            # max-heap (lưu -x)
        self.hi = []            # min-heap
        self.delayed = defaultdict(int)
        self.lo_size = 0
        self.hi_size = 0

    def _prune(self, heap):
        # Dọn các phần tử "đã chết" đang nằm trên đỉnh heap
        while heap:
            x = -heap[0] if heap is self.lo else heap[0]
            if self.delayed[x] > 0:
                self.delayed[x] -= 1
                heapq.heappop(heap)
            else:
                break

    def _balance(self):
        if self.lo_size > self.hi_size + 1:
            heapq.heappush(self.hi, -heapq.heappop(self.lo))
            self.lo_size -= 1; self.hi_size += 1
            self._prune(self.lo)
        elif self.lo_size < self.hi_size:
            heapq.heappush(self.lo, -heapq.heappop(self.hi))
            self.hi_size -= 1; self.lo_size += 1
            self._prune(self.hi)

    def add(self, x):
        if not self.lo or x <= -self.lo[0]:
            heapq.heappush(self.lo, -x); self.lo_size += 1
        else:
            heapq.heappush(self.hi, x); self.hi_size += 1
        self._balance()

    def remove(self, x):
        self.delayed[x] += 1
        if x <= -self.lo[0]:
            self.lo_size -= 1
            if x == -self.lo[0]:
                self._prune(self.lo)
        else:
            self.hi_size -= 1
            if x == self.hi[0]:
                self._prune(self.hi)
        self._balance()

    def median(self):
        if self.k % 2 == 1:
            return float(-self.lo[0])
        return (-self.lo[0] + self.hi[0]) / 2

def median_two_heaps(nums, k):
    dh = DualHeap(k)
    for x in nums[:k]:
        dh.add(x)
    res = [dh.median()]
    for i in range(k, len(nums)):
        dh.add(nums[i])
        dh.remove(nums[i - k])
        res.append(dh.median())
    return res

print(median_two_heaps([1, 3, -1, -3, 5, 3, 6, 7], 3))
assert median_two_heaps([1, 3, -1, -3, 5, 3, 6, 7], 3) == [1.0, -1.0, -1.0, 3.0, 5.0, 6.0]
assert median_two_heaps([1, 2, 3, 4, 2, 3, 1, 4, 2], 3) == [2.0, 3.0, 3.0, 3.0, 2.0, 3.0, 2.0]
assert median_two_heaps([1, 4, 2, 3], 4) == [2.5]

[1.0, -1.0, -1.0, 3.0, 5.0, 6.0]


**Một chi tiết tinh tế:** sau khi `_balance` chuyển một phần tử, đỉnh của heap **nhận** có thể chưa sạch? Không: ta chỉ đẩy vào heap nhận một phần tử *sống* (vì đỉnh heap cho luôn sạch trước khi chuyển), nên đỉnh mới hoặc là phần tử vừa đẩy (sống) hoặc là đỉnh cũ (đã sạch). Còn heap **cho** thì mất đỉnh, phần tử lộ ra có thể là rác → phải `_prune`.

Vì sao đỉnh của cả hai heap luôn sạch trước khi đọc? Mọi thao tác làm thay đổi đỉnh (pop trong balance, đánh dấu xóa đúng phần tử đỉnh) đều kèm `_prune` ngay sau đó.

**Code nộp LeetCode (Cách 3):** (gom lớp `DualHeap` vào trong file)

In [14]:
class SolutionHeap:
    def medianSlidingWindow(self, nums: List[int], k: int) -> List[float]:
        dh = DualHeap(k)
        for x in nums[:k]:
            dh.add(x)
        res = [dh.median()]
        for i in range(k, len(nums)):
            dh.add(nums[i])
            dh.remove(nums[i - k])
            res.append(dh.median())
        return res

print(SolutionHeap().medianSlidingWindow([1, 2, 3, 4, 2, 3, 1, 4, 2], 3))

[2.0, 3.0, 3.0, 3.0, 2.0, 3.0, 2.0]


**Độ phức tạp Cách 3:** mỗi phần tử được push tối đa vài lần và pop tối đa vài lần, mỗi lần `O(log n)` (heap có thể chứa rác nên kích thước tối đa `O(n)`, `log n` và `log k` cùng bậc). Time `O(n log n)`, Space `O(n)` trong trường hợp xấu (rác tích lũy).

### 7.3. Cách 4 — `SortedList` (thư viện `sortedcontainers`)

LeetCode cài sẵn `sortedcontainers`. `SortedList` là danh sách luôn sắp xếp, hỗ trợ `add`, `remove`, truy cập chỉ số đều khoảng `O(log n)`. Code ngắn như Cách 2 nhưng nhanh như Cách 3.

Máy bạn có thể chưa cài thư viện này, nên cell dưới dùng `try/except`: nếu không có, ta tự dựng một bản `SortedList` mini (bọc `bisect`) chỉ để demo cùng giao diện.

In [15]:
try:
    from sortedcontainers import SortedList
    print("Đang dùng sortedcontainers thật")
except ImportError:
    import bisect
    class SortedList:
        """Bản mini cùng giao diện (add/remove/__getitem__), chỉ để demo."""
        def __init__(self, it=()):
            self._a = sorted(it)
        def add(self, x):
            bisect.insort(self._a, x)
        def remove(self, x):
            self._a.pop(bisect.bisect_left(self._a, x))
        def __getitem__(self, i):
            return self._a[i]
        def __len__(self):
            return len(self._a)
    print("Không có sortedcontainers — dùng bản mini")

def median_sortedlist(nums, k):
    sl = SortedList(nums[:k])
    res = [(sl[k // 2] + sl[(k - 1) // 2]) / 2]
    for i in range(k, len(nums)):
        sl.remove(nums[i - k])
        sl.add(nums[i])
        res.append((sl[k // 2] + sl[(k - 1) // 2]) / 2)
    return res

assert median_sortedlist([1, 3, -1, -3, 5, 3, 6, 7], 3) == [1.0, -1.0, -1.0, 3.0, 5.0, 6.0]
print(median_sortedlist([1, 2, 3, 4, 2, 3, 1, 4, 2], 3))

Không có sortedcontainers — dùng bản mini
[2.0, 3.0, 3.0, 3.0, 2.0, 3.0, 2.0]


**Code nộp LeetCode (Cách 4):**

```python
from sortedcontainers import SortedList

class Solution:
    def medianSlidingWindow(self, nums: List[int], k: int) -> List[float]:
        sl = SortedList(nums[:k])
        res = [(sl[k // 2] + sl[(k - 1) // 2]) / 2]
        for i in range(k, len(nums)):
            sl.remove(nums[i - k])
            sl.add(nums[i])
            res.append((sl[k // 2] + sl[(k - 1) // 2]) / 2)
        return res
```

**Độ phức tạp Cách 4:** Time `O(n log k)` (khấu trừ), Space `O(k)`.

## 8. TRACE từng bước

Ta trace **Cách 3 (hai heap)** trên ví dụ 1: `nums = [1,3,-1,-3,5,3,6,7]`, `k = 3`. Các cột:

- `bước`: thao tác vừa làm.
- `lo (vật lý)`, `hi (vật lý)`: nội dung thật của từng heap (sắp xếp để dễ nhìn), **có thể còn rác**.
- `size logic`: số phần tử sống của mỗi nửa.
- `rác`: các giá trị trong `delayed` > 0.
- `median`: trung vị sau khi xử lý xong cửa sổ.

Bảng tính tay (rút gọn):

| i | vào | ra | lo sống | hi sống | median |
|---|---|---|---|---|---|
| khởi tạo | 1,3,-1 | – | {-1,1} | {3} | 1 |
| 3 | -3 | 1 | {-3,-1} | {3} | -1 |
| 4 | 5 | 3 | {-3,-1} | {5} | -1 |
| 5 | 3 | -1 | {-3,3} | {5} | 3 |
| 6 | 6 | -3 | {3,5} | {6} | 5 |
| 7 | 7 | 3 | {5,6} | {7} | 6 |

In [16]:
def trace_two_heaps(nums, k):
    dh = DualHeap(k)

    def show(step):
        junk = {x: c for x, c in dh.delayed.items() if c > 0}
        lo_phys = sorted(-x for x in dh.lo)   # nội dung vật lý (có thể còn rác)
        hi_phys = sorted(dh.hi)
        print(f"{step:<20} lo(vật lý)={lo_phys!s:<14} hi(vật lý)={hi_phys!s:<10} "
              f"size logic=({dh.lo_size},{dh.hi_size}) rác={junk}")

    for x in nums[:k]:
        dh.add(x); show(f"add {x}")
    print("  -> median =", dh.median())
    for i in range(k, len(nums)):
        dh.add(nums[i]); show(f"i={i} add {nums[i]}")
        dh.remove(nums[i - k]); show(f"i={i} remove {nums[i-k]}")
        print("  -> median =", dh.median())

trace_two_heaps([1, 3, -1, -3, 5, 3, 6, 7], 3)

add 1                lo(vật lý)=[1]            hi(vật lý)=[]         size logic=(1,0) rác={}
add 3                lo(vật lý)=[1]            hi(vật lý)=[3]        size logic=(1,1) rác={}
add -1               lo(vật lý)=[-1, 1]        hi(vật lý)=[3]        size logic=(2,1) rác={}
  -> median = 1.0
i=3 add -3           lo(vật lý)=[-3, -1]       hi(vật lý)=[1, 3]     size logic=(2,2) rác={}
i=3 remove 1         lo(vật lý)=[-3, -1]       hi(vật lý)=[3]        size logic=(2,1) rác={}
  -> median = -1.0
i=4 add 5            lo(vật lý)=[-3, -1]       hi(vật lý)=[3, 5]     size logic=(2,2) rác={}
i=4 remove 3         lo(vật lý)=[-3, -1]       hi(vật lý)=[5]        size logic=(2,1) rác={}
  -> median = -1.0
i=5 add 3            lo(vật lý)=[-3, -1]       hi(vật lý)=[3, 5]     size logic=(2,2) rác={}
i=5 remove -1        lo(vật lý)=[-3, 3]        hi(vật lý)=[5]        size logic=(2,1) rác={}
  -> median = 3.0
i=6 add 6            lo(vật lý)=[-3, 3]        hi(vật lý)=[5, 6]     size logic=(2,2) rác

Chú ý dòng `remove`: có lúc phần tử bị xóa nằm sâu trong heap nên **chỉ ghi vào `rác`**, kích thước logic giảm nhưng heap vật lý vẫn còn nó. Đến khi nó lên đỉnh thì mới bị pop thật.

Trace thêm **Cách 2** cho `k` chẵn để thấy trung bình hai phần tử giữa:

In [17]:
def trace_bisect(nums, k):
    window = sorted(nums[:k])
    print(f"khởi tạo window={window} median={(window[k//2] + window[(k-1)//2]) / 2}")
    for i in range(k, len(nums)):
        out, inn = nums[i - k], nums[i]
        pos = bisect.bisect_left(window, out)
        window.pop(pos)
        bisect.insort(window, inn)
        m = (window[k // 2] + window[(k - 1) // 2]) / 2
        print(f"i={i}: bỏ {out:>3} (vị trí {pos}), thêm {inn:>3} -> window={window} median={m}")

trace_bisect([4, 1, 7, 2, 2, 9, 0], 4)

khởi tạo window=[1, 2, 4, 7] median=3.0
i=4: bỏ   4 (vị trí 2), thêm   2 -> window=[1, 2, 2, 7] median=2.0
i=5: bỏ   1 (vị trí 0), thêm   9 -> window=[2, 2, 7, 9] median=4.5
i=6: bỏ   7 (vị trí 2), thêm   0 -> window=[0, 2, 2, 9] median=2.0


## 9. Code nộp LeetCode

Bản nộp chính dùng **hai heap + xóa trễ**, tự chứa, không cần thư viện ngoài, đúng signature `medianSlidingWindow(self, nums, k)`.

In [18]:
from typing import List
import heapq
from collections import defaultdict

class Solution:
    def medianSlidingWindow(self, nums: List[int], k: int) -> List[float]:
        lo, hi = [], []                 # lo: max-heap (lưu -x), hi: min-heap
        delayed = defaultdict(int)
        sizes = [0, 0]                  # sizes[0] = |lo| logic, sizes[1] = |hi| logic

        def prune(heap, sign):
            while heap and delayed[sign * heap[0]] > 0:
                delayed[sign * heap[0]] -= 1
                heapq.heappop(heap)

        def balance():
            if sizes[0] > sizes[1] + 1:
                heapq.heappush(hi, -heapq.heappop(lo))
                sizes[0] -= 1; sizes[1] += 1
                prune(lo, -1)
            elif sizes[0] < sizes[1]:
                heapq.heappush(lo, -heapq.heappop(hi))
                sizes[1] -= 1; sizes[0] += 1
                prune(hi, 1)

        def add(x):
            if not lo or x <= -lo[0]:
                heapq.heappush(lo, -x); sizes[0] += 1
            else:
                heapq.heappush(hi, x); sizes[1] += 1
            balance()

        def remove(x):
            delayed[x] += 1
            if x <= -lo[0]:
                sizes[0] -= 1
                if x == -lo[0]:
                    prune(lo, -1)
            else:
                sizes[1] -= 1
                if x == hi[0]:
                    prune(hi, 1)
            balance()

        def median():
            return float(-lo[0]) if k % 2 else (-lo[0] + hi[0]) / 2

        for x in nums[:k]:
            add(x)
        res = [median()]
        for i in range(k, len(nums)):
            add(nums[i])
            remove(nums[i - k])
            res.append(median())
        return res

sol = Solution()
assert sol.medianSlidingWindow([1, 3, -1, -3, 5, 3, 6, 7], 3) == [1.0, -1.0, -1.0, 3.0, 5.0, 6.0]
assert sol.medianSlidingWindow([1, 2, 3, 4, 2, 3, 1, 4, 2], 3) == [2.0, 3.0, 3.0, 3.0, 2.0, 3.0, 2.0]
assert sol.medianSlidingWindow([2147483647, 2147483647], 2) == [2147483647.0]
print("Solution OK")

Solution OK


## 10. Vì sao thuật toán đúng

**Cách 2 (bisect).** Bất biến: trước mỗi lần ghi trung vị, `window` chứa đúng multiset `nums[i-k+1..i]` và đã sắp xếp.
- Khởi tạo: `sorted(nums[:k])` — đúng.
- Bước trượt: `bisect_left(window, out)` trả về vị trí một bản sao của `out` (chắc chắn tồn tại vì `out` thuộc cửa sổ cũ) → pop xóa đúng một bản sao. `insort` giữ thứ tự. Multiset mới đúng là cửa sổ mới.
- Đọc `window[k//2]`, `window[(k-1)//2]` chính là định nghĩa trung vị.

**Cách 3 (hai heap).** Gọi `L`, `H` là multiset phần tử **sống** trong `lo`, `hi`. Bất biến sau mỗi `add/remove + balance`:
1. `L ∪ H` = cửa sổ hiện tại (trong quá trình trượt có lúc là `k+1` phần tử, sau `remove` trở lại `k`).
2. `max(L) <= min(H)`.
3. `|L| - |H| ∈ {0, 1}`, và `lo_size = |L|`, `hi_size = |H|`.
4. Đỉnh `lo`, `hi` là phần tử sống.

- `add(x)`: đặt `x` vào nửa phù hợp giữ (2). `balance` chuyển phần tử **lớn nhất của L** sang H (hoặc nhỏ nhất của H sang L) — chuyển phần tử "biên" thì (2) vẫn giữ.
- `remove(x)`: `x` là phần tử sống. Nếu `x <= max(L)` thì có một bản sao của `x` trong `L` (nếu không, mọi bản sao ở `H` nghĩa là `x >= min(H) >= max(L)`, kết hợp `x <= max(L)` → `x = max(L) = min(H)`, khi đó coi như xóa bản ở `L` và đổi tên một bản ở `H` sang `L` — giá trị giống hệt nên multiset hai nửa về giá trị vẫn hợp lệ). Ta giảm size đúng nửa, đánh dấu chờ xóa. `balance` khôi phục (3).
- Từ (2)(3): với `k` lẻ, `|L| = (k+1)/2` và `max(L)` là phần tử thứ `(k+1)/2` — đúng trung vị; với `k` chẵn, `max(L)` và `min(H)` là hai phần tử giữa.

Một điểm cần yên tâm: `delayed[x]` được tiêu thụ khi *bất kỳ* bản sao `x` lên đỉnh. Vì các bản sao có cùng giá trị, việc xóa bản nào cũng như nhau — chỉ cần **tổng số** bản sống của từng giá trị đúng.

## 11. Phân tích Time/Space

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Sắp xếp lại | `O((n-k+1)·k log k)` | `O(k)` | TLE với input lớn |
| 2. bisect | `O(n·k)` lý thuyết | `O(k)` | dời mảng bằng C, thường AC |
| 3. Hai heap + lazy | `O(n log n)` | `O(n)` | rác có thể tích lũy |
| 4. SortedList | `O(n log k)` | `O(k)` | cần thư viện |

Kết quả có `n-k+1` số thực → bộ nhớ đầu ra `O(n)` (không tính vào space phụ).

Đo thời gian thực tế trên input vừa phải:

In [19]:
import time, random
random.seed(7)
big = [random.randint(-2**31, 2**31 - 1) for _ in range(20000)]
for name, f in [("bisect", median_bisect), ("two_heaps", median_two_heaps), ("sortedlist", median_sortedlist)]:
    t = time.perf_counter()
    f(big, 5000)
    print(f"{name:<11}: {time.perf_counter() - t:.3f}s")

bisect     : 0.013s
two_heaps  : 0.016s
sortedlist : 0.014s


## 12. Bộ test

Gồm: ví dụ đề, edge case (`k = 1`, `k = n`, số trùng, số cực trị `±2^31`, `k` chẵn), và kiểm thử ngẫu nhiên có seed so với brute force.

In [20]:
def close(a, b):
    return len(a) == len(b) and all(abs(x - y) < 1e-5 for x, y in zip(a, b))

methods = {
    "brute": median_brute,
    "bisect": median_bisect,
    "two_heaps": median_two_heaps,
    "sortedlist": median_sortedlist,
    "Solution": lambda a, k: Solution().medianSlidingWindow(a, k),
}

cases = [
    (([1, 3, -1, -3, 5, 3, 6, 7], 3), [1.0, -1.0, -1.0, 3.0, 5.0, 6.0]),
    (([1, 2, 3, 4, 2, 3, 1, 4, 2], 3), [2.0, 3.0, 3.0, 3.0, 2.0, 3.0, 2.0]),
    (([5], 1), [5.0]),                                        # n = k = 1
    (([1, 2, 3], 1), [1.0, 2.0, 3.0]),                        # k = 1
    (([1, 2, 3, 4], 4), [2.5]),                               # k = n, chẵn
    (([2, 2, 2, 2], 2), [2.0, 2.0, 2.0]),                     # toàn trùng
    (([-2**31, 2**31 - 1], 2), [-0.5]),                       # cực trị
    (([2**31 - 1] * 3, 2), [2147483647.0, 2147483647.0]),
    (([1, 1, 1, 2, 2, 2], 3), [1.0, 1.0, 2.0, 2.0]),
]
for (arr, k), exp in cases:
    for name, f in methods.items():
        got = f(list(arr), k)
        assert close(got, exp), (name, arr, k, got, exp)
print("Tất cả ví dụ & edge case: OK")

Tất cả ví dụ & edge case: OK


In [21]:
import random
random.seed(2026)
for trial in range(400):
    n = random.randint(1, 30)
    k = random.randint(1, n)
    arr = [random.randint(-5, 5) for _ in range(n)]   # giá trị nhỏ -> nhiều số trùng, dễ lộ lỗi
    exp = median_brute(arr, k)
    for name, f in methods.items():
        assert close(f(list(arr), k), exp), (name, arr, k)
for trial in range(100):
    n = random.randint(1, 200)
    k = random.randint(1, n)
    arr = [random.randint(-2**31, 2**31 - 1) for _ in range(n)]
    exp = median_brute(arr, k)
    for name, f in methods.items():
        assert close(f(list(arr), k), exp), (name, k)
print("500 test ngẫu nhiên khớp brute force")

500 test ngẫu nhiên khớp brute force


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Dùng `list.remove` cho heap (làm hỏng cấu trúc heap)

**Code sai:**

```python
lo.remove(-out)          # xóa giữa heap
# quên heapify -> heap không còn hợp lệ, đỉnh có thể sai
```

**Vì sao sai:** `list.remove` dời phần tử, phá vỡ tính chất heap. Dù gọi `heapq.heapify` lại thì mỗi lần tốn `O(k)` — mất lợi thế.

**Code đúng:** dùng xóa trễ (`delayed[out] += 1`, giảm size logic, dọn đỉnh khi cần) như Cách 3.

### Lỗi 2 — Dùng `len(heap)` thay cho kích thước logic

In [22]:
# Minh họa: len(lo) tính cả rác nên cân bằng sai
import heapq
lo = [-5, -3, -1]; heapq.heapify(lo)
delayed = {3: 1}           # số 3 đã bị xóa nhưng còn nằm trong heap
print("len(lo) =", len(lo), "nhưng số phần tử sống chỉ là", len(lo) - sum(delayed.values()))
# Code đúng: luôn dùng lo_size / hi_size do ta tự quản lý

len(lo) = 3 nhưng số phần tử sống chỉ là 2


### Lỗi 3 — Trung vị dùng phép chia nguyên `//`

**Code sai:** `(a + b) // 2` → với `a=1, b=2` ra `1` thay vì `1.5`; với số âm `(-1 + -2) // 2 = -2` (làm tròn xuống) thay vì `-1.5`.

**Code đúng:** `(a + b) / 2`.

In [23]:
a, b = -1, -2
print("sai:", (a + b) // 2, " đúng:", (a + b) / 2)

sai: -2  đúng: -1.5


### Lỗi 4 — Thứ tự thêm/xóa sai làm heap rỗng khi xóa

Nếu `k = 1` và ta **xóa trước rồi mới thêm**, sau khi xóa `lo` rỗng logic; `remove` lại so `x <= -lo[0]` với đỉnh có thể là rác. Thêm trước rồi xóa (như Cách 3) đảm bảo cửa sổ luôn có ít nhất 1 phần tử sống khi xác định nửa chứa `out`.

In [24]:
# Kiểm chứng: với k=1, thứ tự add-rồi-remove chạy đúng
print(median_two_heaps([4, -2, 9, 9, 0], 1))
assert median_two_heaps([4, -2, 9, 9, 0], 1) == [4.0, -2.0, 9.0, 9.0, 0.0]

[4.0, -2.0, 9.0, 9.0, 0.0]


### Lỗi 5 — Lệch chỉ số khi trượt

**Code sai:** `for i in range(k, len(nums)+1): window.pop(...nums[i-k]...); insort(window, nums[i])` → `IndexError` ở vòng cuối hoặc thiếu trung vị đầu tiên.

**Code đúng:** ghi trung vị cửa sổ đầu **trước** vòng lặp, rồi `for i in range(k, len(nums))` xử lý vào `nums[i]`, ra `nums[i-k]`, rồi ghi trung vị. Tổng số trung vị phải là `n - k + 1` — luôn `assert` điều này.

## 14. Mẹo & nhận dạng pattern

- Thấy **"median"** + **dữ liệu thay đổi** (stream hoặc cửa sổ trượt) → nghĩ ngay **hai heap** (LeetCode 295 Find Median from Data Stream là bài "mẹ").
- Thấy **"cửa sổ trượt" + cần thống kê theo thứ tự** (min/max/median/k-th) → cần cấu trúc **thêm + xóa + truy vấn**: SortedList, hai heap + lazy, deque đơn điệu (chỉ cho min/max).
- Heap cần **xóa phần tử bất kỳ** → **lazy deletion** với bộ đếm `delayed`.
- Python + LeetCode: `sortedcontainers.SortedList` là "vũ khí bí mật" cho mọi bài cần cây cân bằng.
- Luôn tự hỏi: *"Mỗi bước trượt chỉ thay đổi 2 phần tử — mình có đang làm lại toàn bộ không?"*

## 15. Biến thể

1. **Find Median from Data Stream (LC 295):** chỉ thêm, không xóa → hai heap không cần lazy.
2. **Sliding Window Maximum (LC 239):** chỉ cần max → **deque đơn điệu** `O(n)`.
3. **K-th smallest trong cửa sổ:** thay chỉ số `k//2` bằng `t-1` trong SortedList.
4. **Trung bình cửa sổ:** chỉ cần tổng chạy `O(n)` — so sánh để thấy vì sao median khó hơn.
5. **Trung vị có trọng số / trung vị tối thiểu hóa tổng khoảng cách** (Sliding Window, LC 462 liên quan).

Demo biến thể 3 — phần tử nhỏ thứ `t` trong mỗi cửa sổ:

In [25]:
def kth_smallest_sliding(nums, k, t):
    window = sorted(nums[:k])
    res = [window[t - 1]]
    for i in range(k, len(nums)):
        window.pop(bisect.bisect_left(window, nums[i - k]))
        bisect.insort(window, nums[i])
        res.append(window[t - 1])
    return res

print(kth_smallest_sliding([1, 3, -1, -3, 5, 3, 6, 7], 3, 1))   # min mỗi cửa sổ
assert kth_smallest_sliding([1, 3, -1, -3, 5, 3, 6, 7], 3, 3) == [3, 3, 5, 5, 6, 7]  # max

[-1, -3, -3, -3, 3, 3]


Demo biến thể 1 — median của dòng dữ liệu (không xóa):

In [26]:
class MedianFinder:
    def __init__(self):
        self.lo, self.hi = [], []
    def addNum(self, x):
        heapq.heappush(self.lo, -x)
        heapq.heappush(self.hi, -heapq.heappop(self.lo))
        if len(self.hi) > len(self.lo):
            heapq.heappush(self.lo, -heapq.heappop(self.hi))
    def findMedian(self):
        if len(self.lo) > len(self.hi):
            return float(-self.lo[0])
        return (-self.lo[0] + self.hi[0]) / 2

mf = MedianFinder()
out = []
for x in [5, 15, 1, 3]:
    mf.addNum(x); out.append(mf.findMedian())
print(out)
assert out == [5.0, 10.0, 5.0, 4.0]

[5.0, 10.0, 5.0, 4.0]


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute | Sắp xếp lại mỗi cửa sổ | `O(n·k log k)` | `O(k)` | Kiểm chứng, input nhỏ |
| 2. bisect | Giữ list sắp xếp, pop + insort | `O(n·k)` | `O(k)` | Python nhanh gọn, phỏng vấn cần giải thích hạn chế |
| 3. Hai heap + lazy | Max-heap nửa dưới, min-heap nửa trên, xóa trễ | `O(n log n)` | `O(n)` | Lời giải chuẩn, ngôn ngữ không có cây cân bằng |
| 4. SortedList | Cây/khối sắp xếp có chỉ số | `O(n log k)` | `O(k)` | Python trên LeetCode, code ngắn nhất |

## 17. Bài tập

Làm theo thứ tự dễ → khó. Các cell TODO **không được gọi** khi chạy notebook; hãy tự bỏ comment để thử.

**Bài 1 (đoán output).** `median_brute([4, 4, 1, 7], 2)` in ra gì? Đoán trước khi xem đáp án.

**Bài 2 (tính tay).** Với `nums = [3, 8, 1, 6, 2]`, `k = 4`, viết hai cửa sổ, sắp xếp, tính trung vị. Sau đó trace Cách 3: sau khi khởi tạo, `lo` và `hi` sống chứa gì?

**Bài 3 (điền TODO).** Hoàn thiện hàm `median_bisect_todo` bên dưới.

In [27]:
def median_bisect_todo(nums, k):
    window = sorted(nums[:k])
    res = []
    # TODO: ghi trung vị cửa sổ đầu
    for i in range(k, len(nums)):
        # TODO: xóa nums[i-k] bằng bisect_left + pop
        # TODO: chèn nums[i] bằng insort
        # TODO: ghi trung vị
        pass
    return res

# Bỏ comment để tự kiểm tra sau khi làm xong:
# assert median_bisect_todo([1,3,-1,-3,5,3,6,7], 3) == [1.0,-1.0,-1.0,3.0,5.0,6.0]

**Bài 4 (tự viết lại).** Không nhìn mục 9, tự viết lại lớp hai heap với xóa trễ. Kiểm tra bằng bộ test ngẫu nhiên ở mục 12.

**Bài 5 (mini challenge).** Viết `sliding_window_range_median(nums, k)` trả về danh sách `max - median` của mỗi cửa sổ (độ lệch từ trung vị đến max). Gợi ý: cần cả max lẫn median — SortedList cho cả hai.

In [28]:
def sliding_window_range_median_todo(nums, k):
    # TODO: trả về [max(window) - median(window) for mỗi cửa sổ]
    raise NotImplementedError

# Không gọi hàm TODO khi chạy notebook.

## 18. Gợi ý 3 mức

**Bài 1–2**
- Mức 1: Liệt kê từng cửa sổ trước.
- Mức 2: `k` chẵn → trung bình hai phần tử giữa.
- Mức 3: Bài 1: cửa sổ `[4,4]`, `[4,1]`, `[1,7]`.

**Bài 3**
- Mức 1: Viết hàm `med()` đọc `window[k//2]` và `window[(k-1)//2]`.
- Mức 2: Xóa trước, chèn sau (hoặc ngược lại đều được với list sắp xếp).
- Mức 3: Ghi trung vị đầu trước vòng lặp, cuối mỗi vòng ghi trung vị mới.

**Bài 4**
- Mức 1: Ba thành phần: `lo`, `hi`, `delayed` + hai size logic.
- Mức 2: Mọi lần đỉnh thay đổi theo hướng "lộ phần tử mới" → `prune`.
- Mức 3: Thêm trước, xóa sau; `remove` so `x` với `-lo[0]`.

**Bài 5**
- Mức 1: max của cửa sổ sắp xếp là phần tử cuối `window[-1]`.
- Mức 2: Dùng lại khung Cách 2.
- Mức 3: `res.append(window[-1] - med())`.

## 19. Đáp án tham khảo

In [29]:
# Bài 1
ans1 = median_brute([4, 4, 1, 7], 2)
print("Bài 1:", ans1)
assert ans1 == [4.0, 2.5, 4.0]

# Bài 2
ans2 = median_brute([3, 8, 1, 6, 2], 4)
print("Bài 2:", ans2)       # [3,8,1,6]->[1,3,6,8]->4.5 ; [8,1,6,2]->[1,2,6,8]->4.0
assert ans2 == [4.5, 4.0]
dh = DualHeap(4)
for x in [3, 8, 1, 6]:
    dh.add(x)
print("lo sống:", sorted(-x for x in dh.lo), "hi sống:", sorted(dh.hi))

Bài 1: [4.0, 2.5, 4.0]
Bài 2: [4.5, 4.0]
lo sống: [1, 3] hi sống: [6, 8]


In [30]:
# Bài 3
def median_bisect_sol(nums, k):
    window = sorted(nums[:k])
    med = lambda: (window[k // 2] + window[(k - 1) // 2]) / 2
    res = [med()]
    for i in range(k, len(nums)):
        window.pop(bisect.bisect_left(window, nums[i - k]))
        bisect.insort(window, nums[i])
        res.append(med())
    return res

assert median_bisect_sol([1, 3, -1, -3, 5, 3, 6, 7], 3) == [1.0, -1.0, -1.0, 3.0, 5.0, 6.0]

# Bài 4: dùng lại class Solution (mục 9) — đã kiểm tra ở mục 12.

# Bài 5
def sliding_window_range_median(nums, k):
    window = sorted(nums[:k])
    med = lambda: (window[k // 2] + window[(k - 1) // 2]) / 2
    res = [window[-1] - med()]
    for i in range(k, len(nums)):
        window.pop(bisect.bisect_left(window, nums[i - k]))
        bisect.insort(window, nums[i])
        res.append(window[-1] - med())
    return res

def range_median_brute(nums, k):
    out = []
    for i in range(len(nums) - k + 1):
        s = sorted(nums[i:i + k])
        out.append(s[-1] - (s[k // 2] + s[(k - 1) // 2]) / 2)
    return out

random.seed(5)
for _ in range(200):
    n = random.randint(1, 25); k = random.randint(1, n)
    arr = [random.randint(-9, 9) for _ in range(n)]
    assert sliding_window_range_median(arr, k) == range_median_brute(arr, k)
print("Đáp án bài 3, 5: OK", sliding_window_range_median([1, 3, -1, -3, 5, 3, 6, 7], 3))

Đáp án bài 3, 5: OK [2.0, 4.0, 6.0, 2.0, 1.0, 1.0]


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template hai heap + xóa trễ:**

```python
lo, hi, delayed, sz = [], [], defaultdict(int), [0, 0]   # lo lưu -x
def prune(h, sgn):
    while h and delayed[sgn*h[0]]:
        delayed[sgn*h[0]] -= 1; heappop(h)
def balance():
    if sz[0] > sz[1] + 1: heappush(hi, -heappop(lo)); sz[0]-=1; sz[1]+=1; prune(lo, -1)
    elif sz[0] < sz[1]:   heappush(lo, -heappop(hi)); sz[1]-=1; sz[0]+=1; prune(hi, 1)
def add(x):    (push lo nếu x <= -lo[0] else hi); balance()
def remove(x): delayed[x]+=1; (giảm size nửa chứa x; prune nếu x là đỉnh); balance()
median = -lo[0] nếu k lẻ, (-lo[0] + hi[0]) / 2 nếu k chẵn
```

**Template SortedList:** `sl.remove(nums[i-k]); sl.add(nums[i]); (sl[k//2] + sl[(k-1)//2]) / 2`

**Checklist trước khi submit:**

- [ ] Số trung vị trả về đúng `n - k + 1`.
- [ ] Dùng `/` (float), không dùng `//`.
- [ ] Đã thử `k = 1`, `k = n`, `k` chẵn, số trùng, số `±2^31`.
- [ ] Heap: dùng **size logic**, không dùng `len(heap)`.
- [ ] Heap: sau mọi `heappop` ở balance và sau khi đánh dấu xóa phần tử đỉnh → `prune`.
- [ ] Thứ tự: thêm phần tử mới **trước**, xóa phần tử cũ **sau** (tránh heap rỗng khi `k = 1`).
- [ ] Đối chiếu với brute force trên input ngẫu nhiên có nhiều số trùng.